In [1]:
# Cell 1: Setup and Data Loading
import sys
from pathlib import Path
import pandas as pd
import plotly.express as px

# Automatically find the project root (one folder up from the notebook) and add it to sys.path
project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

# Now Python can find the etl module
from etl.fetch_buoy import fetch_buoy_swell

# Fetch the last 30 days of buoy data, using cache to save time on reruns
df_buoy = fetch_buoy_swell(days=30, use_cache=True)

# Preview the data
display(df_buoy.head())
display(df_buoy.info())

,timestamp_utc,station,WVHT,DPD,MWD,APD
0,2026-05-23 01:26:00+00:00,46232,1.0,13,201,7.3
1,2026-05-23 00:56:00+00:00,46232,1.0,10,275,7.2
2,2026-05-23 00:26:00+00:00,46232,1.0,14,194,7.1
3,2026-05-22 23:56:00+00:00,46232,1.0,13,181,7.2
4,2026-05-22 23:26:00+00:00,46232,1.0,12,221,7.3


<class 'pandas.core.frame.DataFrame'>
Index: 2749 entries, 0 to 3434
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   timestamp_utc  2749 non-null   datetime64[ns, UTC]
 1   station        2749 non-null   object             
 2   WVHT           2749 non-null   float64            
 3   DPD            2749 non-null   int64              
 4   MWD            2749 non-null   int64              
 5   APD            2749 non-null   float64            
dtypes: datetime64[ns, UTC](1), float64(2), int64(2), object(1)
memory usage: 150.3+ KB


None

In [ ]:
import numpy as np

# 1. Feature Engineering: Swell & Wind Categories
# Swell Energy Proxy (Height squared * Period)
df_buoy['swell_energy'] = (df_buoy['WVHT'] ** 2) * df_buoy['DPD']

# Categorize Dominant Period (DPD)
dpd_conditions = [
    (df_buoy['DPD'] <= 7),
    (df_buoy['DPD'] > 7) & (df_buoy['DPD'] <= 11),
    (df_buoy['DPD'] > 11) & (df_buoy['DPD'] <= 14),
    (df_buoy['DPD'] > 14)
]
dpd_choices = ['Weak', 'Moderate', 'Good', 'Excellent']

df_buoy['dpd_category'] = np.select(dpd_conditions, dpd_choices, default='Unknown')

# 2. Break-Specific Filtering (e.g., Blacks Beach Alignment)
# West/NW swell (270-300°), Mid-size+ (1.5m+), Long Period (12s+)
blacks_ideal = df_buoy[
    (df_buoy['MWD'].between(270, 300)) & 
    (df_buoy['WVHT'] >= 1.5) &
    (df_buoy['DPD'] >= 12)
]

# La Jolla Shores Alignment
# South/SW swell (180-220°), Mid-size+ (1.5m+), Long Period (12s+)
lajolla_ideal = df_buoy[
    (df_buoy['MWD'].between(180, 220)) & 
    (df_buoy['WVHT'] >= 1.5) &
    (df_buoy['DPD'] >= 12)
]

# PB Point Alignment
# West/NW swell (270-310°), Mid-size+ (1.5m+), Long Period (12s+)
pb_ideal = df_buoy[
    (df_buoy['MWD'].between(270, 310)) & 
    (df_buoy['WVHT'] >= 1.5) &
    (df_buoy['DPD'] >= 12)
]

# 3. Domain-Specific Visualizations

# Polar plot to see where the largest, longest period swells come from
fig_polar = px.scatter_polar(
    df_buoy, r="WVHT", theta="MWD", color="DPD",
    title="Swell Energy by Direction (Look for 270-300° for Blacks Beach)",
    color_continuous_scale="viridis"
)
fig_polar.show()

# Scatter plot checking the relationship between DPD and WVHT
# Colored by our engineered dpd_category since surfline_rating is not merged yet
fig_quality = px.scatter(
    df_buoy, 
    x="DPD", 
    y="WVHT", 
    color="dpd_category",
    title="Wave Quality: Period vs Height (Expect powerful waves top-right)",
    category_orders={
        "dpd_category": ["Weak", "Moderate", "Good", "Excellent"]
    }
)
fig_quality.show()


In [5]:
import numpy as np
import plotly.express as px

# 1. Evaluate the conditions for all three breaks
conditions = [
    # 270-300° is optimal for BOTH Blacks and PB Point
    (df_buoy['MWD'].between(270, 300)) & (df_buoy['WVHT'] >= 1.5) & (df_buoy['DPD'] >= 12),
    # 300-310° is optimal for PB Point, but NOT Blacks
    (df_buoy['MWD'].between(300, 310)) & (df_buoy['WVHT'] >= 1.5) & (df_buoy['DPD'] >= 12),
    # 180-220° is strictly La Jolla Shores
    (df_buoy['MWD'].between(180, 220)) & (df_buoy['WVHT'] >= 1.5) & (df_buoy['DPD'] >= 12)
]

# The labels corresponding to the conditions above
choices = ['Blacks & PB Point', 'PB Point Only', 'La Jolla Shores']

# Create the new column, defaulting to 'Sub-optimal' if none of the conditions are met
df_buoy['optimal_break'] = np.select(conditions, choices, default='Sub-optimal / Other')

# 2. Plot the newly categorized data
fig_polar = px.scatter_polar(
    df_buoy, 
    r="WVHT", 
    theta="MWD", 
    color="optimal_break", 
    size="DPD", # Shifting DPD to dot size so we still see the wave period!
    title="Ideal Swell Alignment by Break",
    color_discrete_map={
        'Blacks & PB Point': '#EF553B',    # Red
        'PB Point Only': '#FFA15A',        # Orange
        'La Jolla Shores': '#00CC96',      # Green
        'Sub-optimal / Other': '#E5ECF6'   # Light Gray (fades into background)
    }
)

# Optional: Force the radius to start at 0 so the visual scale makes sense
fig_polar.update_layout(polar=dict(radialaxis=dict(rangemode="tozero")))
fig_polar.show()

In [6]:
# 1. APD Analysis: The "Cleanliness" Gap
df_buoy['period_gap'] = df_buoy['DPD'] - df_buoy['APD']

fig_gap = px.histogram(
    df_buoy, x='period_gap', color='dpd_category',
    title='Swell Cleanliness: DPD vs APD Gap (Larger gap = Cleaner swell)',
    category_orders={"dpd_category": ["Weak", "Moderate", "Good", "Excellent"]}
)
fig_gap.show()

# 2. Time-Series EDA: Tracking incoming swells over the last 30 days
fig_time = px.line(
    df_buoy, x='timestamp_utc', y='WVHT', color='station',
    title='Wave Height Trends Over Time'
)
fig_time.show()

# 3. Station Comparison EDA: Point Loma vs Mission Bay
fig_station = px.box(
    df_buoy, x='station', y='WVHT', color='station',
    title='Wave Height Distribution by Buoy Station'
)
fig_station.show()